# TorchRef Code Examples

This notebook demonstrates the code examples from the TorchRef README, providing a comprehensive overview of the library's capabilities.

## Setup

First, let's import TorchRef and set up our file paths.

In [ ]:
import torchref
import torch
from torchref import ROOT_TORCHREF # Root directort of torchref installation

directory_notebook = f'{ROOT_TORCHREF}/example_notebooks/'

# File paths - update these to your data files
mtzpath = f'{directory_notebook}/1DAW.mtz'
pdbpath = f'{directory_notebook}/1DAW.pdb'

/tmp/ipykernel_1962443/37228406.py:2: UserWarning: TorchRef auto-configured 4 threads. Set TORCHREF_NUM_THREADS to override.
  import torchref


## Basic functionality

Use a model to compute structure factor and scale them to a dataset

In [ ]:
from torchref import ReflectionData
from torchref import ModelFT
from torchref import Scaler

from torchref.math_functions.math_torch import get_rfactors
# The ReflectionData class can load MTZ files or CIF files
# It by default loads intensities and converts them to structure factors using French-Wilson conversion
# It keeps track of flagged values, rfree flags, and other useful information

data = ReflectionData().load_mtz(mtzpath)

model = ModelFT().load_pdb(pdbpath)

hkl, F, sigF, rfree = data()


# Calculate structure factors for the the model for a given set of hkl
fcalc = model(hkl)

scale = Scaler(model, data)

scale.initialize() # setup initial scale factors, and also bulk solvent if applicable
scale.refine_lbfgs() # refine scale factors using L-BFGS optimizer

scaled_fcalc = scale(fcalc)

Fcalc_abs = torch.abs(scaled_fcalc)

rwork, rfree = get_rfactors(F, Fcalc_abs, rfree)

print(f"Rwork: {rwork:.3f}, Rfree: {rfree:.3f}")


FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (360, 144, 108)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 20 bins.
Calculating initial scale factors using 20 bins.
Refining scales with LBFGS...
Scale refinement complete. rwork: 0.2105, rfree: 0.2750

Final Scale Parameters: 
  log_scale: tensor([-6.0531, -5.9564, -5.9575, -5.9123, -5.9025, -5.9140, -5.8207, -5.7930,
        -5.7878, -5.7477, -5.7135, -5.6629, -5.6040, -5.5500, -5.5008, -5.4291,
        -5.3401, -5.3197, -5.3258, -5.3439

/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


## Model parameters can be selectively frozen

This may be used during refinement or when only parts of the model want to be refined

In [20]:
model = ModelFT().load_pdb(pdbpath)

print([i.shape for i in model.parameters()])

model.freeze('b') # Freeze all B-factor parameters
print([i.shape for i in model.parameters()])

model.unfreeze('b') # Unfreeze all B-factor parameters
print([i.shape for i in model.parameters()])

model.freeze('all') # Freeze all parameters
print([i.shape for i in model.parameters()])

model.unfreeze('all') # Unfreeze all parameters
print([i.shape for i in model.parameters()])

model.freeze('xyz') # Freeze all position parameters

print([i.shape for i in model.parameters()])
model.unfreeze('xyz') # Unfreeze all position parameters

# We can also freeze unfreeze only a part of the model through phenix style selections

selection_str = "chain A and resseq 10:20"

model.freeze_selection(selection_str)

print([i.shape for i in model.parameters()])

# And similarly

model.freeze_all() # First freeze all

model.unfreeze_selection(selection_str) # Then unfreeze selection

print([i.shape for i in model.parameters()])

model.unfreeze_selection("all")  # Unfreeze all again, if we made a selection before we have to do this to unfreeze all

# we can also do this at a parameter level

mask_xyz = torch.zeros(model.xyz.shape, dtype=torch.bool)  # just to illustrate
mask_xyz[1:500,:] = True  # example mask to unfreeze atoms 1 to 500

model.xyz.update_refinable_mask(mask_xyz)

print([i.shape for i in model.parameters()])



Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (360, 144, 108)
  ✓ Using direct indexing (no interpolation)
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051, 3]), torch.Size([3])]
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051]), torch.Size([3])]
Selection 'chain A and resseq 10:20' (91 atoms) frozen for xyz
  Total refinable atoms for xyz: 2960/3051
  Applied mask to xyz: 2960 atoms refinable
Selection 'chain A and resseq 10:20' (91 atoms) frozen for b
  Total refinable atoms for b: 2960/3051
  Applied mask to b: 2960 atoms refinable
Selection 'chain A and resseq 10:20' (91 atoms) frozen for

## Basic Refinement

### Loading Data and Model Components Separately

In [2]:
from torchref.io import ReflectionData
from torchref.model import ModelFT
from torchref.scaling import Scaler

# Load reflection data
data = ReflectionData(verbose=1)
data.load_mtz(mtzpath)

# Load model with structure factor calculation capability
model = ModelFT()
model.load_pdb(pdbpath)

# Create scaler
scaler = Scaler(model, data)

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (360, 144, 108)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 20 bins.


### Initialize Refinement from Files Automatically

In [3]:
from torchref.refinement import LBFGSRefinement

# Initialize refinement directly from files
refinement = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

print(f"Initial R-factors: {refinement.get_rfactor()}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (160, 72, 54)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 10 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
------------------------------------------------

/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


### Manual Refinement Initialization

Refinement can also be initialized manually by setting components individually.

In [4]:
from torchref.refinement import LBFGSRefinement
from torchref.io import ReflectionData
from torchref.model import ModelFT
from torchref.scaling import Scaler
from torchref.restraints import Restraints

# Create components
data_manual = ReflectionData()
data_manual.load_mtz(mtzpath)

model_manual = ModelFT()
model_manual.load_pdb(pdbpath)

scaler_manual = Scaler(model_manual, data_manual)

restraints_manual = Restraints(model_manual)

# Manual initialization
refinement_manual = LBFGSRefinement()
refinement_manual.model = model_manual
refinement_manual.reflection_data = data_manual
refinement_manual.scaler = scaler_manual
refinement_manual.restraints = restraints_manual

# Initialize standard targets
refinement_manual._init_targets()

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (360, 144, 108)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 20 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
----------------------------------------------

### Running Refinement

In [5]:
# Shake coordinates to simulate a starting model with errors
refinement.model.shake_coords(0.1)
print(f"R-factors after shaking: {refinement.get_rfactor()}")

# Run refinement (refines all parameters, including scales)
refinement.refine_everything(macro_cycles=3)

print(f"Final R-factors: {refinement.get_rfactor()}")

R-factors after shaking: (0.23415739834308624, 0.29191628098487854)

LBFGS Refinement Everything - Cycle 1/3


/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


Calculating initial scale factors using 10 bins.
Outlier detection: 60/23356 (0.26%) outliers found
  Log-ratio statistics: mean=-0.025, std=0.593
  Z-score threshold: 5.0
Outlier detection: 60 reflections flagged as outliers out of 23356.
Refining scales with LBFGS...
Scale refinement complete. rwork: 0.2346, rfree: 0.2927

Final Scale Parameters: 
  log_scale: tensor([-3.3494, -3.2703, -3.2585, -3.2251, -3.2107, -3.1826, -3.1078, -3.0608,
        -3.0006, -3.0808])
  U: tensor([-0.1767, -0.0868, -0.0061, -0.0009, -0.1569, -0.0047])
  solvent.log_k_solvent: -0.9970752596855164
  solvent.b_solvent: 45.989253997802734
  solvent.phase_offset: 0.03305055573582649
Outlier detection: 84/23356 (0.36%) outliers found
  Log-ratio statistics: mean=0.064, std=0.553
  Z-score threshold: 5.0
Outlier detection: 84 reflections flagged as outliers out of 23356.
After scaling: Rwork=0.2340, Rfree=0.2935
Refining scales with LBFGS...
Scale refinement complete. rwork: 0.2339, rfree: 0.2934

Final Scale 

### Writing Output Files

In [6]:
# Write refined structure
refinement.write_out_pdb(f"{directory_notebook}/refined.pdb")

# Write structure factors
refinement.write_out_mtz(f"{directory_notebook}/refined.mtz")

Added map coefficients:
  2mFo-DFc: FWT, PHWT
  mFo-DFc: DELFWT, PHDELWT
  Resolution range: 2.05 - 69.56 Å
✓ Wrote MTZ file: /das/work/units/LBR-FEL/p17490/Peter/Library/torchref/example_notebooks//refined.mtz
  Reflections: 23356
  Columns: H, K, L, F-obs, SIGF-obs, I-obs, SIGI-obs, R-free-flags, 2FOFCWT, PH2FOFCWT, FOFCWT, PHFOFCWT, F-model, PH-model


## Custom Target Functions

One of TorchRef's key strengths is the ease of defining custom refinement targets. Thanks to PyTorch's automatic differentiation, you simply define the forward computation.

In [7]:
import torch
from torchref.refinement.targets import Target

class CustomTarget(Target):
    """Custom refinement target with automatic gradient computation."""
    name = 'test_target'

    def __init__(self, refinement, weight=1.0):
        super().__init__(refinement)

    def forward(self):
        # Define your target function - gradients computed automatically!
        F_calc = self.refinement.model.get_F_calc()
        F_obs = self.refinement.reflection_data.F

        # Custom loss computation (simple least squares example)
        loss = torch.mean((torch.abs(F_calc) - F_obs) ** 2)
        return loss

### Registering Custom Targets

Custom targets are registered in the LossState of the refinement. This breaks the circular dependency and allows easy integration with weighting schemes.

In [8]:
# Create a fresh refinement for this example
refinement_custom = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

# Create loss state
loss_state = refinement_custom.create_loss_state()

test_target = CustomTarget(refinement_custom)
name = test_target.name
# Register the custom target
loss_state.register_target(name, test_target)

# Set the weight of the custom target
loss_state.set_weight(name, 3.0)

print(f"Registered targets: {list(loss_state.targets.keys())}")
print(f"Weights: {loss_state.weights}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (160, 72, 54)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 10 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
------------------------------------------------

## GPU Acceleration

Structure factor calculations parallelize naturally on GPUs. Move everything to GPU for significant speedups.

In [9]:
# Check if CUDA is available
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.get_device_name(0)}")
    
    # Create refinement on GPU
    refinement_gpu = LBFGSRefinement(
        data_file=mtzpath,
        pdb=pdbpath,
    )
    
    # Move to GPU
    refinement_gpu.cuda()
    
    # Create loss state and move to GPU
    loss_state_gpu = refinement_gpu.create_loss_state()
    loss_state_gpu.cuda()
    
    print(f"Model device: {refinement_gpu.model.xyz.device}")
    print(f"LossState device: {loss_state_gpu.device}")
else:
    print("CUDA not available, running on CPU")

CUDA available: Tesla T4
FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (160, 72, 54)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 10 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
-----------------------

## Saving and Loading State

TorchRef supports full `state_dict` for saving and loading complete refinement states.

In [10]:
import torch
from torchref.refinement import LBFGSRefinement

# Create refinement
refinement_save = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

# Save complete refinement state
# torch.save(refinement_save.state_dict(), "checkpoint.pt")

# Load state into new refinement object
# new_refinement = LBFGSRefinement()
# new_refinement.load_state_dict(torch.load("checkpoint.pt"))

print("State dict keys:", list(refinement_save.state_dict().keys())[:10], "...")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (160, 72, 54)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 10 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
------------------------------------------------

## Integration with Machine Learning

TorchRef's PyTorch foundation enables seamless integration with neural networks. Here's an example of a hybrid model combining crystallographic refinement with a neural network.

In [11]:
import torch
import torch.nn as nn
from torchref.refinement import LBFGSRefinement
from torchref.model.simple_model import SimpleModel

dataset = ReflectionData()
dataset.load_mtz(mtzpath)

spacegroup = dataset.spacegroup
cell = dataset.cell

class HybridModel(nn.Module):
    """Combine crystallographic refinement with a neural network."""
    
    def __init__(self, refinement):
        super().__init__()
        self.refinement = refinement
        self.neural_prior = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)  # Predicts coordinates and b factors
        )
        self.model = SimpleModel()

    def forward(self, features, cell, spacegroup, hkl):
        # Neural network predictions
        coords_b = self.neural_prior(features)
        coords = coords_b[:, :3]
        b_factors = coords_b[:, 3]
        occs = torch.ones_like(b_factors)
        return self.model(xyz=coords, b=b_factors, occ=occs, cell=cell, spacegroup=spacegroup)

        
# This is just a demonstration of how something like this could be done.
# In practice we would also need some kind of alignment and scaling which is WIP. 

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged


## Automatic Differentiation Advantage

Traditional refinement programs require explicit implementation of gradients. TorchRef eliminates this burden using PyTorch's autograd.

In [12]:
# Demonstrate with actual refinement
demo_refinement = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

# Compute loss - parameters already have requires_grad=True
loss_state = demo_refinement.create_loss_state()
demo_refinement.add_target_info_to_state(loss_state)
demo_refinement.populate_state_meta(loss_state)
demo_refinement.update_weights(loss_state)

loss = loss_state.aggregate()
print(f"Loss: {loss.item():.4f}")

# Gradients computed automatically via PyTorch autograd!
loss.backward()

# Access gradients through refinement.parameters()
params = list(demo_refinement.parameters())
grads_computed = sum(1 for p in params if p.grad is not None)
total_grad_norm = sum(p.grad.norm().item() for p in params if p.grad is not None)
print(f"Parameters with gradients: {grads_computed}/{len(params)}")
print(f"Total gradient norm: {total_grad_norm:.4f}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (160, 72, 54)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 10 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
------------------------------------------------

## Loss State Management

The LossState object is central to TorchRef's refinement workflow. It manages targets, weights, and metadata.

In [13]:
# Create refinement
ref = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

# Create the loss state for refinement
loss_state = ref.create_loss_state()

# Add target info to the loss state needed for updating weights
ref.add_target_info_to_state(loss_state)

# Populate the state with meta info also needed for weights
ref.populate_state_meta(loss_state)

# Update/set weights in the loss state
ref.update_weights(loss_state)

loss = loss_state.aggregate()

loss.backward()

print(ref.model.xyz.refinable_params.grad)
print(ref.model.b.refinable_params.grad)
print(ref.model.occupancy.refinable_params.grad)


FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged


Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (160, 72, 54)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 10 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disul

## Summary

This notebook demonstrated:

1. **Basic Refinement** - Loading data, models, and running refinement
2. **Custom Targets** - Defining and registering custom loss functions
3. **GPU Acceleration** - Moving computations to CUDA devices
4. **State Management** - Saving and loading refinement checkpoints
5. **ML Integration** - Combining neural networks with crystallographic refinement
6. **Automatic Differentiation** - PyTorch's autograd for gradient computation
7. **LossState** - Managing targets, weights, and metadata

For more examples, see the `examples/` directory in the TorchRef repository.